Notes(What needs to be done): 
1. 
2. 
etc

# 02 - Cleaned Data

**Purpose:** Build the modeling datasets from raw. Reshapes to the two grains
required downstream, fixes types, and validates the result. No analysis, no
plots, no interpretation.

**Input:** `data/raw/CASComp Main DataSet.csv` (unmodified)

**Outputs:**

- `data/processed/frequency_data.csv` - one row per student-coverage (40,000 rows),
  with `claim_count` and `total_amount`. Input to the frequency models.
- `data/processed/claims.csv` - one row per claim (1,819 rows). Input to the
  severity models.

**Depends on:** 00_Inventory, 01_Data_Checks

**This is the only notebook that writes to `data/processed/`.** Everything from
03 onward reads these files and never touches raw. Re-running this notebook
should reproduce both outputs exactly.

**Transformations applied**


**Decisions carried from 01**

- No deductible found, so losses are treated as ground-up with no adjustment
- No censoring at policy limits, so severity is uncensored

**Author:** Dalton | **Last run:** 2026-09-14

Load data and constants

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
df = pd.read_csv(RAW / "CASComp Main DataSet.csv")

Create frequency and claims data frames

In [2]:
policy_cols = ['student_id','name','class','study','gpa','greek','off_campus',
               'distance_to_campus','gender','sprinklered','risk_tier','training_holdout']

freq = (df.assign(is_claim=(df.claim_id > 0).astype(int))
          .groupby(policy_cols + ['coverage'], as_index=False, observed=True)
          .agg(claim_count=('is_claim','sum'),
               total_amount=('amount','sum')))

claims = df[df.claim_id > 0].copy()


print(claims.shape)
display(freq.head())
display(claims.head())



(1819, 15)


,student_id,name,class,study,gpa,greek,off_campus,distance_to_campus,gender,sprinklered,risk_tier,training_holdout,coverage,claim_count,total_amount
0,1,"Saiz, Leslie",Junior,Humanities,1.203913,Non-greek,On campus,0.0,Male,True,2,training,Additional Living Expense,0,0.0
1,1,"Saiz, Leslie",Junior,Humanities,1.203913,Non-greek,On campus,0.0,Male,True,2,training,Guest Medical,0,0.0
2,1,"Saiz, Leslie",Junior,Humanities,1.203913,Non-greek,On campus,0.0,Male,True,2,training,Liability,0,0.0
3,1,"Saiz, Leslie",Junior,Humanities,1.203913,Non-greek,On campus,0.0,Male,True,2,training,Personal Property,0,0.0
4,2,"Reddy, Dawna",Junior,Science,0.609616,Non-greek,On campus,0.0,Female,True,2,training,Additional Living Expense,0,0.0


,student_id,name,class,study,gpa,greek,off_campus,distance_to_campus,gender,sprinklered,coverage,claim_id,amount,risk_tier,training_holdout
40,11,"Leigh, Shelley",Sophomore,Other,0.981003,Non-greek,Off campus,10.083255,Female,True,Personal Property,1,990.596391,1,training
84,22,"Stark, Najee",Senior,Science,1.874571,Greek,On campus,0.000000,Male,True,Personal Property,2,1172.942616,3,training
104,27,"Roy, Dylan",Sophomore,Business,2.582975,Greek,On campus,0.000000,Male,True,Personal Property,3,2106.469682,1,training
112,29,"Mora, Yesica",Senior,Business,2.448340,Non-greek,Off campus,0.568290,Female,True,Liability,4,6943.265254,3,training
128,33,"Ignacio, Austin",Senior,Business,2.843812,Non-greek,On campus,0.000000,Did not respond,True,Guest Medical,5,7197.105825,3,training


-

write to csv for future use

In [3]:
freq.to_csv(PROC / "frequency_data.csv", index=False)
claims.to_csv(PROC / "claims.csv", index=False)

print(sorted(p.name for p in PROC.iterdir()))

['claims.csv', 'frequency_data.csv']
